# Simulating the UCJ ansatz with fermionic backpropagation

This guide demonstrates how to simulate a one-layer [UCJ ansatz](../explanations/lucj.ipynb) using the algorithm described [here](https://arxiv.org/abs/2607.21337). This method can be used to efficiently simulate any one-layer UCJ ansatz exactly, with an optional final orbital rotation. It computes the expectation value of an electronic Hamiltonian in the Heisenberg picture in $O(N^7)$ time for $N$ spatial orbitals.

In [1]:
import pyscf
import pyscf.cc
import scipy

import ffsim

## LUCJ ansatz for a closed-shell molecule
We'll construct the ansatz for a nitrogen molecule in the 6-31g basis set. Since it's a closed-shell system, use the spin-balanced UCJ ansatz. We will restrict the pair connectivity, however fermionic backpropagation in general will work for any connectivity.

In [2]:
# Build N2 molecule
mol = pyscf.gto.Mole()
mol.build(
    atom=[["N", (0, 0, -0.5)], ["N", (0, 0, 0.5)]],
    basis="6-31g",
    symmetry="Dooh",
)

# Define active space
n_frozen = 2
active_space = range(n_frozen, mol.nao_nr())

# Get molecular data and Hamiltonian
scf = pyscf.scf.RHF(mol).run()
mol_data = ffsim.MolecularData.from_scf(scf, active_space=active_space)
norb, nelec = mol_data.norb, mol_data.nelec
mol_hamiltonian = mol_data.hamiltonian
print(f"norb = {norb}")
print(f"nelec = {nelec}")

# Get CCSD t2 amplitudes for initializing the ansatz
ccsd = pyscf.cc.CCSD(
    scf, frozen=[i for i in range(mol.nao_nr()) if i not in active_space]
).run()

n_reps = 1

# Define interactions
pairs_aa = [(p, p + 1) for p in range(norb - 1)]
pairs_ab = None

ucj_op = ffsim.UCJOpSpinBalanced.from_t_amplitudes(
    ccsd.t2,
    t1=ccsd.t1,
    n_reps=n_reps,
    interaction_pairs=(pairs_aa, pairs_ab),
    optimize=True,
    regularization=0.1,
    options=dict(maxiter=100),
)

converged SCF energy = -108.835236570774


norb = 16
nelec = (5, 5)
E(CCSD) = -109.0398256929733  E_corr = -0.2045891221988314


Now, let's simulate the ansatz using the backpropagation method. 

In [3]:
energy = ffsim.ucj_energy_spin_balanced(ucj_op, mol_hamiltonian, nelec)
print(f"HF energy: {scf.e_tot:.6f}")
print(f"LUCJ Energy: {energy:.6f}")
print(f"CCSD energy: {ccsd.e_tot:.6f}")

HF energy: -108.835237
LUCJ Energy: -108.910134
CCSD energy: -109.039826


We can also variationally optimize the ansatz parameters to achieve lower ground state energies. This is particularly useful for strongly correlated systems where the CCSD parameters may not be optimal. `ucj_energy_and_grad_func_spin_balanced` returns a callable that computes the energy and its gradient with respect to the ansatz parameters using JAX for autodifferentiation. We pass this callable directly to `scipy.optimize.minimize`. Note that for these examples, the number of terms is manageable on a GPU, but for larger systems, GPU acceleration might necessitate chunking the tensors, which can be set using the `chunk_size` argument.

In [4]:
def callback(intermediate_result: scipy.optimize.OptimizeResult):
    print(f" {intermediate_result.fun:.6f}")


value_and_grad = ffsim.ucj_energy_and_grad_func_spin_balanced(
    mol_hamiltonian,
    nelec,
    interaction_pairs=(pairs_aa, pairs_ab),
    with_final_orbital_rotation=ucj_op.final_orbital_rotation is not None,
    chunk_size=256,
)
result = scipy.optimize.minimize(
    value_and_grad,
    ucj_op.to_parameters(interaction_pairs=(pairs_aa, pairs_ab)),
    method="L-BFGS-B",
    jac=True,
    callback=callback,
    options=dict(maxiter=5),
)
optimal_ucj = ffsim.UCJOpSpinBalanced.from_parameters(
    result.x,
    norb=norb,
    n_reps=n_reps,
    interaction_pairs=(pairs_aa, pairs_ab),
    with_final_orbital_rotation=ucj_op.final_orbital_rotation is not None,
)
energy = ffsim.ucj_energy_spin_balanced(optimal_ucj, mol_hamiltonian, nelec)
print(f"Optimized LUCJ energy: {energy:.6f}")

 -108.932491


 -108.946318


 -108.953896


 -108.960281


 -108.964077


Optimized LUCJ energy: -108.964077


## UCJ ansatz for an open-shell molecule
We'll use a hydroxyl radical in the 6-31g basis set as an example of an open-shell system. For this example, we'll use unrestricted pair connectivity, i.e. the full UCJ ansatz.

In [5]:
# Build HO molecule
mol = pyscf.gto.Mole()
mol.build(
    atom=[["H", (0, 0, 0)], ["O", (0, 0, 1.0)]],
    basis="6-31g",
    spin=1,
    # ROHF fails to converge with Coov, so use the C2v subgroup
    symmetry="C2v",
)

# Get molecular data and Hamiltonian
scf = pyscf.scf.ROHF(mol).run()
mol_data = ffsim.MolecularData.from_scf(scf)
norb, nelec = mol_data.norb, mol_data.nelec
mol_hamiltonian = mol_data.hamiltonian
print(f"norb = {norb}")
print(f"nelec = {nelec}")

# Get CCSD t2 amplitudes for initializing the ansatz
ccsd = pyscf.cc.CCSD(scf).run()

ucj_op = ffsim.UCJOpSpinUnbalanced.from_t_amplitudes(
    ccsd.t2,
    t1=ccsd.t1,
    n_reps=1,
    optimize=True,
    regularization=0.1,
    options=dict(maxiter=100),
)

converged SCF energy = -75.3608533074516


norb = 11
nelec = (5, 4)

WARN: RCCSD method does not support ROHF method. ROHF object is converted to UHF object and UCCSD method is called.



E(UCCSD) = -75.46256067414772  E_corr = -0.101707366696122


In [6]:
energy = ffsim.ucj_energy_spin_unbalanced(ucj_op, mol_hamiltonian, nelec)
print(f"HF energy: {scf.e_tot:.6f}")
print(f"UCJ Energy: {energy:.6f}")
print(f"CCSD energy: {ccsd.e_tot:.6f}")

HF energy: -75.360853
UCJ Energy: -75.401747
CCSD energy: -75.462561


In [7]:
value_and_grad = ffsim.ucj_energy_and_grad_func_spin_unbalanced(
    mol_hamiltonian,
    nelec,
    with_final_orbital_rotation=ucj_op.final_orbital_rotation is not None,
    chunk_size=256,
)
result = scipy.optimize.minimize(
    value_and_grad,
    ucj_op.to_parameters(),
    method="L-BFGS-B",
    jac=True,
    callback=callback,
    options=dict(maxiter=5),
)
optimal_ucj = ffsim.UCJOpSpinUnbalanced.from_parameters(
    result.x,
    norb=norb,
    n_reps=1,
    with_final_orbital_rotation=ucj_op.final_orbital_rotation is not None,
)
energy = ffsim.ucj_energy_spin_unbalanced(optimal_ucj, mol_hamiltonian, nelec)
print(f"Optimized UCJ energy: {energy:.6f}")

 -75.435090


 -75.437998


 -75.440210


 -75.441318


 -75.442156
Optimized UCJ energy: -75.442156
